In [ ]:
#LLM CALLING
import os
from dotenv import load_dotenv

# Read the .env file so its values become available to the code
load_dotenv()

# Pull out the two values we need for Open WebUI
api_key = os.getenv("OPENWEBUI_API_KEY")
base_url = os.getenv("OPENWEBUI_BASE_URL")

# Quick check that both loaded, without printing the secret key itself
print("Key loaded:", api_key is not None)
print("Address:", base_url)

In [ ]:
import requests

# Ask Open WebUI for the list of models this key can use.
# Same pattern as the SEC API requests: an address, a header with your key, a timeout.
models_response = requests.get(
    f"{base_url}/models",
    headers={"Authorization": f"Bearer {api_key}"},   # "Bearer" is the standard way to send a key
    timeout=10,
)
print(models_response.status_code)   # 200 means it worked

# Print the ID of each model; you'll copy one of these into the next cell
for model in models_response.json()["data"]:
    print(model["id"])

In [ ]:
import anthropic

# The client is your connection to the API, like 'con' was for DuckDB.
# We point it at Open WebUI instead of going to Anthropic directly.
client = anthropic.Anthropic(
    api_key=api_key,     # your ID badge
    base_url=base_url,   # where to send requests
)

In [ ]:
import time

# A setting, not a secret, so it lives in code for now. It moves to a config file in Phase 6.
# Paste one Claude model ID from the list Cell 2 printed
MODEL = "anthropic.claude-sonnet-5-5"

start = time.perf_counter()   # Start a stopwatch to measure latency

# Send one message in the Anthropic Messages format
response = client.messages.create(
    model=MODEL,
    max_tokens=50,   # Cap the reply length (this also caps the cost)
    messages=[{"role": "user", "content": "Reply with exactly: gateway works"}],,
)

elapsed = time.perf_counter() - start   # Stop the stopwatch

print(response.content[0].text)    # The model's reply text
print(f"Latency: {elapsed:.2f}s")  # How long the round trip took
print(response.usage)              # Token counts, which we'll use for cost tracking